In [1]:
import os
from dotenv import load_dotenv

from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_core.messages import HumanMessage

from langgraph.prebuilt import create_react_agent
from langgraph.graph.state import CompiledStateGraph

from langchain_openai import ChatOpenAI

load_dotenv()

True

In [2]:
# establish db connection 

r"C:\Users\Marius.Gnoth\projects\HorseRacing\data\processed\horse_racing_data.db"
db_path = os.getenv("SQLITE_DB_PATH", r"C:\Users\Marius.Gnoth\projects\HorseRacing\data\processed\horse_racing_data.db")

db = SQLDatabase.from_uri(f"sqlite:///{db_path}")
print(db.dialect)

sqlite


In [3]:
llm = ChatOpenAI(
    model="gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [6]:
toolkit = SQLDatabaseToolkit(
    db = db,
    llm = llm
).get_tools()

horses_db_agent = create_react_agent(
    model = llm,
    tools = toolkit,
    name="HorseRacingDataAnalyst",
    prompt = """You are a Horse Racing Data Analysis Agent.""",
)

In [7]:
example_query = [
    HumanMessage(
        content="""How many unique horses are in the database?""")
    ]

response = horses_db_agent.invoke(
    {
        "messages": example_query
    }
)

for m in response["messages"]:
    m.pretty_print() 

================================ Human Message =================================

How many unique horses are in the database?
================================== Ai Message ==================================
Name: HorseRacingDataAnalyst
Tool Calls:
  sql_db_list_tables (call_2aU0MuF2V6IdSe95vHtJ2JgI)
 Call ID: call_2aU0MuF2V6IdSe95vHtJ2JgI
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

horse_racing_data
================================== Ai Message ==================================
Name: HorseRacingDataAnalyst
Tool Calls:
  sql_db_schema (call_eafPUO0e2BtJpt826D6Ej8nO)
 Call ID: call_eafPUO0e2BtJpt826D6Ej8nO
  Args:
    table_names: horse_racing_data
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE horse_racing_data (
	track_id TEXT, 
	race_date TEXT, 
	race_number INTEGER, 
	program_number TEXT, 
	trakus_index INTEGER, 
	latitude REAL, 
	longitude REAL